In [3]:
pip install langdetect nltk pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 7.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993332 sha256=c4f84e466a6473f90b66a4c78cba15a935b908b6122b4e10b64d886ddb28f9c3
  Stored in directory: /Users/Khoai/Library/Caches/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
import glob
import os
import re
from langdetect import detect, LangDetectException
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /Users/Khoai/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
DATA_RAW = "../Dataset/data_new_approach"

In [8]:
csv_paths = glob.glob(os.path.join(DATA_RAW, "*.csv"))

In [9]:
print("CSV files found:")
for p in csv_paths:
    print(" ", os.path.basename(p))

CSV files found:
  marcus_by_goldman_sachs__combined_us.csv
  chase__combined_us.csv
  bank_of_america__combined_us.csv
  wells_fargo__combined_us.csv
  citi__combined_us.csv


In [10]:
def load_csv(path):
    df = pd.read_csv(path, dtype=str)  # all columns as string
    df["source_file"] = os.path.basename(path)
    return df

raw = pd.concat([load_csv(p) for p in csv_paths], ignore_index=True)
print(f"\nTotal rows: {len(raw)}")


Total rows: 5450


In [11]:
print(raw.dtypes)
print("\nReview per platform: ")
print(raw["platform"].value_counts().rename("n_reviews"))
print("\nMissing values: ")
missing = raw.isnull().sum().reset_index()
missing.columns = ["variable", "n_missing"]
print(missing.sort_values("n_missing", ascending=False))
print("\nMissing by platform")
print(raw.groupby("platform").apply(lambda x: x.isnull().sum()))

platform         object
storefront       object
app_id           object
review_id        object
date             object
user             object
rating           object
title            object
review           object
version          object
package          object
thumbsUpCount    object
appVersion       object
source_file      object
dtype: object

Review per platform: 
platform
google_play    3000
app_store      2450
Name: n_reviews, dtype: int64

Missing values: 
         variable  n_missing
2          app_id       3000
7           title       3000
9         version       3000
12     appVersion       2676
10        package       2450
11  thumbsUpCount       2450
0        platform          0
1      storefront          0
3       review_id          0
4            date          0
5            user          0
6          rating          0
8          review          0
13    source_file          0

Missing by platform
             platform  storefront  app_id  review_id  date  user  rating  

/var/folders/ld/rwg8qntx0sg34zkdmmh8mcc80000gp/T/ipykernel_17467/983431939.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(raw.groupby("platform").apply(lambda x: x.isnull().sum()))


The missing data is mainly the difference information that appear on Google Store and App Store

In [13]:
raw["raw_text"] = (
    raw["title"].fillna("") + " " + raw["review"].fillna("")
).str.strip()

# Word count (mirrors str_count(\\S+))
raw["raw_length_words"] = raw["raw_text"].str.split().str.len()
print("\n--- raw_length_words summary ---")
print(raw["raw_length_words"].describe())


--- raw_length_words summary ---
count    5450.000000
mean       22.044587
std        29.047547
min         1.000000
25%         4.000000
50%        11.000000
75%        28.000000
max       377.000000
Name: raw_length_words, dtype: float64


In [15]:
#standatdize bank name
def extract_bank_name(filename):
    name = filename.replace("__combined_us.csv", "")
    name = name.replace("_", " ")
    return name.title()

df1 = raw.copy()
df1["bank"] = df1["source_file"].apply(extract_bank_name)
df1["platform"] = df1["platform"].map({
    "app_store": "iOS",
    "google_play": "Android"
}).fillna(df1["platform"])

print("\nReviews by bank and platform ")
print(df1.groupby(["bank", "platform"]).size().reset_index(name="n"))



Reviews by bank and platform 
                      bank platform    n
0          Bank Of America  Android  600
1          Bank Of America      iOS  490
2                    Chase  Android  600
3                    Chase      iOS  490
4                     Citi  Android  600
5                     Citi      iOS  490
6  Marcus By Goldman Sachs  Android  600
7  Marcus By Goldman Sachs      iOS  490
8              Wells Fargo  Android  600
9              Wells Fargo      iOS  490


In [17]:
#fix data type
df2 = df1.copy()
df2["rating"] = pd.to_numeric(df2["rating"], errors="coerce")
df2["date"] = pd.to_datetime(df2["date"], errors="coerce", utc=True)

print("\nRating summary")
print(df2["rating"].describe())
print("\n Date summary")
print(df2["date"].describe())


Rating summary
count    5450.000000
mean        3.697982
std         1.739445
min         1.000000
25%         1.000000
50%         5.000000
75%         5.000000
max         5.000000
Name: rating, dtype: float64

 Date summary
count                                   2450
mean     2026-01-10 15:49:03.699183616+00:00
min                2024-06-06 21:25:49+00:00
25%      2025-12-24 09:06:11.249999872+00:00
50%         2026-03-21 19:18:06.500000+00:00
75%      2026-04-07 20:22:56.750000128+00:00
max                2026-04-20 16:04:39+00:00
Name: date, dtype: object


In [19]:
#keep only English language
def detect_language(text):
    try:
        if not text or len(text.strip()) < 10:
            return "unknown"
        return detect(text)
    except LangDetectException:
        return "unknown"

df2["language"] = df2["raw_text"].apply(detect_language)

print("\nLanguage counts")
print(df2["language"].value_counts())

df_english = df2[df2["language"] == "en"].copy()

print(f"""
Total reviews:        {len(df2)}
English reviews:      {len(df_english)}
Removed:              {len(df2) - len(df_english)}
Pct removed:          {100 * (len(df2) - len(df_english)) / len(df2):.1f}%
""")


Language counts
language
en         4379
unknown     440
es          221
af           54
fr           48
da           39
ca           29
no           27
so           27
it           27
ro           26
tl           25
de           21
nl           17
hr           11
cy            7
et            7
sl            7
sq            7
pt            5
pl            4
cs            4
sk            4
zh-cn         4
id            3
ru            1
sw            1
vi            1
fi            1
sv            1
lv            1
tr            1
Name: count, dtype: int64

Total reviews:        5450
English reviews:      4379
Removed:              1071
Pct removed:          19.7%

